# quantkit — Colab runner

Thin runner only: all logic lives in the `quantkit` package. Runtime → Change runtime type → **L4**.
Run the setup cells, then the section for the phase you're on.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,power.limit --format=csv
import torch; assert 'L4' in torch.cuda.get_device_name(0), 'Not an L4 — switch runtime before collecting numbers'

In [ ]:
# Checkpoints and results persist on Drive; the HF model cache stays on local disk
# (reading a freshly written multi-GB safetensors back through the Drive mount corrupted it).
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/quantkit'
os.makedirs(WORK, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'
# Llama-3.2 is gated: accept the license on its HF page, and store your token in
# Colab Secrets (key icon) as HF_TOKEN. Scripts run as subprocesses and inherit this.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
REPO = 'https://github.com/ankitsharma285/quantkit.git'
!rm -rf /content/quantkit && git clone -q $REPO /content/quantkit
%cd /content/quantkit
!pip install -q -e '.[dev]'

In [ ]:
!python -m pytest -q

In [ ]:
import json, glob

def show(results_dir):
    rows = [json.load(open(f)) for f in sorted(glob.glob(f'{results_dir}/*.json'))]
    print(f"{'run':34s} {'wikitext2':>10s} {'c4':>10s} {'quant s':>8s}")
    for r in rows:
        p = r['perplexity']
        w = p.get('wikitext2', {}).get('ppl', float('nan'))
        c = p.get('c4', {}).get('ppl', float('nan'))
        t = r.get('quant_seconds', float('nan'))
        print(f"{r['label']:34s} {w:10.3f} {c:10.3f} {t:8.1f}")

## Phase 1 — harness + RTN (done)

In [ ]:
R = f'{WORK}/results/day1'
!python scripts/quantize_eval.py --method fp16 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size 128 --save $WORK/ckpt/rtn-g128 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size -1 --results-dir $R
!python scripts/quantize_eval.py --method rtn --bits 3 --group-size 128 --fake --results-dir $R

In [ ]:
# Reload the saved int4 checkpoint: ppl must match the rtn-w4-g128 run exactly.
!python scripts/quantize_eval.py --load $WORK/ckpt/rtn-g128 --datasets wikitext2 --results-dir $R

In [ ]:
show(f'{WORK}/results/day1')

## Phase 2 — GPTQ

**Smoke test first** (16 calibration samples, 8 eval windows, ~1–2 min): catches wiring problems before the full runs.

In [ ]:
!python scripts/quantize_eval.py --method gptq --nsamples 16 --datasets wikitext2 --max-windows 8 \
    --results-dir $WORK/results/sanity

**Main runs.** Same calibration set (C4, 128 × 2048, seed 0) for every run. Expected order on WikiText-2: fp16 9.75 < GPTQ < RTN 11.73.

In [ ]:
R2 = f'{WORK}/results/phase2'
os.makedirs(R2, exist_ok=True)
# Bring the Phase 1 baselines along so the Phase 2 table is self-contained.
!cp "{WORK}"/results/day1/*_fp16.json "{WORK}"/results/day1/*_rtn-w4-g128.json "{R2}"/
!python scripts/quantize_eval.py --method gptq --save $WORK/ckpt/gptq-g128 --results-dir $R2

In [ ]:
# The packed GPTQ checkpoint must reproduce the in-memory perplexity exactly.
!python scripts/quantize_eval.py --load $WORK/ckpt/gptq-g128 --datasets wikitext2 --results-dir $R2

In [ ]:
# Variants: act-order (static groups), 3-bit, and GPTQModel's default damping (0.05 vs the paper's 0.01).
!python scripts/quantize_eval.py --method gptq --act-order --results-dir $R2
!python scripts/quantize_eval.py --method gptq --bits 3 --fake --results-dir $R2
!python scripts/quantize_eval.py --method gptq --damp 0.05 --results-dir $R2

In [ ]:
show(R2)

### Diagnosis: GPTQ without act-order worse than RTN

First run (Nov 19): GPTQ W4 g128 gave 34.2 (reloaded checkpoint, damp 0.01) and 81.6 (the damp 0.05 run, which
overwrote the main JSON because damping wasn't in the label; fixed). Act-order gave 10.78, better than RTN's 11.73.

Hypothesis: the first token of each calibration window is an attention sink with massive activations in a
few hidden channels, and it dominates the Hessians. Test: rerun with those positions left out of the
Hessians, and check how lopsided each Hessian diagonal is.

In [ ]:
R3 = f'{WORK}/results/phase2_diag'
os.makedirs(R3, exist_ok=True)
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --skip-first-tokens 1 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --skip-first-tokens 4 --results-dir $R3
!python scripts/quantize_eval.py --method gptq --datasets wikitext2 --act-order --skip-first-tokens 1 --results-dir $R3
show(R3)

In [ ]:
# Which modules blow up, and how lopsided are their Hessians?
!python scripts/inspect_gptq.py {R3}/*.json

### Diagnosis, round 2: mechanism and noise

Round 1 (WikiText-2): plain GPTQ 34.17; leaving the first token out of the Hessians 11.16 (first 4: 11.10);
act-order 10.78; act-order + skip first token 11.24; RTN 11.73. Vanilla GPTQModel also breaks (50.86).

Open questions:
1. *Mechanism.* Does fixing group scales from the original weights (static groups, no reordering) also
   rescue plain GPTQ? If yes, the failure is dynamic group scales being chosen after error feedback has
   inflated the weights (`scale growth` in the inspect output shows it directly). If no, reordering is what helps.
2. *Noise.* Is act-order's 10.78 vs 11.24 real or calibration-sample luck? Two more calibration seeds.

In [ ]:
R4 = f'{WORK}/results/phase2_diag2'
os.makedirs(R4, exist_ok=True)
run = 'python scripts/quantize_eval.py --method gptq --datasets wikitext2 --results-dir ' + R4
!{run}                                   # rerun plain GPTQ to record scale growth
!{run} --static-groups
!{run} --static-groups --skip-first-tokens 1
!{run} --act-order --calib-seed 1
!{run} --act-order --calib-seed 2
show(R4)

In [ ]:
!python scripts/inspect_gptq.py {R4}/*.json

### Reference: GPTQModel

Run in a **fresh session** (Runtime → Disconnect and delete runtime, then the setup cells): installing GPTQModel
changes packages.

Round 1: vanilla settings (no act-order) gave 50.86 vs quantkit's 34.17. Both are in the broken regime, where
small implementation differences get amplified, so that pair can't test parity. The parity check uses the
configuration that works: act-order with static groups.

In [ ]:
!pip install -q gptqmodel
# GPTQModel imports transformers' AutoProcessor, which imports torchaudio; Colab's torchaudio
# is built for a different CUDA than its torch. quantkit doesn't use audio, so remove it.
!pip uninstall -y -q torchaudio

In [ ]:
R2 = f'{WORK}/results/phase2'
# Matched, healthy comparison: act-order with static groups in both implementations.
# Done when this is within 1% of quantkit's gptq-w4-g128-ao (10.780).
!python scripts/reference_gptqmodel.py --act-order --out /content/ckpt/ref-gptqmodel-ao --results-dir $R2
show(R2)

### Parity gap: where do quantkit and GPTQModel diverge?

Matched act-order run: GPTQModel 11.086 vs quantkit 10.780 on WikiText-2 (2.8%; 6.0% on C4), outside the 1% target.
Compare the two checkpoints module by module, in execution order. Same session as the reference run
(GPTQModel installed). Each step skips itself if its checkpoint already exists.

In [ ]:
import os
if not os.path.exists('/content/ckpt/ref-gptqmodel-ao'):
    !python scripts/reference_gptqmodel.py --act-order --out /content/ckpt/ref-gptqmodel-ao --results-dir /content/tmp_results
if not os.path.exists('/content/ckpt/gptq-ao'):
    !python scripts/quantize_eval.py --method gptq --act-order --datasets wikitext2 --max-windows 4 --save /content/ckpt/gptq-ao --results-dir /content/tmp_results
!python scripts/compare_checkpoints.py /content/ckpt/gptq-ao /content/ckpt/ref-gptqmodel-ao | tee $WORK/results/phase2/compare_ao.txt